# PoC V6.0 — Phases 3 & 4: MPNN Predictor + Dual-Route Deployment

This notebook implements the final two phases of the GNN-HVA v6.0 pipeline:

- **Phase 3**: MPNN training with energy-driven validation (replaces V4 MLP)
- **Phase 4**: Dual-route deployment — Adapt-VQE (main) + QRC (fallback)

System: 1D TFIM, N=6, HVA p=2.

In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path so 'poc.v6' package is importable
_src = str(Path().resolve().parents[1])  # src/poc/v6 -> src/poc -> src
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import torch
import matplotlib.pyplot as plt
import logging

from poc.v6.hamiltonian_builder import HamiltonianBuilder, make_lattice
from poc.v6.classical_solver import ClassicalSolver
from poc.v6.hva_builder import HVACircuitBuilder
from poc.v6.mpnn_predictor import MPNNPredictor, build_graph_dataset, train_mpnn
from poc.v6.qrc_pipeline import QRCPipeline
from poc.v6.hardware_deployer import HardwareDeployer
from poc.v6.pipeline_utils import load_phase12_dataset, assert_observable_locality
from poc.v6.config import VQEConfig

from qiskit.primitives import StatevectorEstimator

logging.basicConfig(level=logging.INFO)
torch.manual_seed(42)
np.random.seed(42)

print('V6.0 Phases 3-4 initialized.')

V6.0 Phases 3-4 initialized.


## Load Phase 1+2 Data (with validation)

In [2]:
data = load_phase12_dataset('phase1_phase2_tfim_N6_p2_v6.npz')

N = int(data['n_qubits'])
J = float(data['J'])
p_layers = int(data['p_layers'])
h_values = data['h_values']
ground_energies = data['ground_energies']
gaps = data['gaps']
theta_opt_all = data['theta_opt']
fidelities = data['fidelities']
mag_x_exact = data['mag_x']
corr_zz_exact = data['corr_zz']

print(f'Loaded: N={N}, p={p_layers}, {len(h_values)} h-points')
print(f'Cost function: {data["cost_function"]}, Version: {data["version"]}')

INFO:poc.v6.pipeline_utils:Dataset loaded: phase1_phase2_tfim_N6_p2_v6.npz (version=v6.0, cost=energy)


Loaded: N=6, p=2, 27 h-points
Cost function: energy, Version: v6.0


## Phase 3: MPNN Training

In [3]:
# Build graph dataset with fidelity filter
FID_THRESHOLD = 0.93
base_lattice = make_lattice('chain_1d', N, J=J, h=1.0)

dataset = build_graph_dataset(
    base_lattice, h_values, theta_opt_all, ground_energies,
    fidelities=fidelities, fidelity_threshold=FID_THRESHOLD,
)
print(f'Training dataset: {len(dataset)} graphs (threshold={FID_THRESHOLD})')

INFO:poc.v6.mpnn_predictor:Built graph dataset: 17/27 points (fidelity threshold=0.93)


Training dataset: 17 graphs (threshold=0.93)


In [4]:
# Energy validation callback
builder = HamiltonianBuilder()
hva_builder = HVACircuitBuilder()
qc, theta_params = hva_builder.create(N, p_layers, base_lattice)
estimator = StatevectorEstimator()

def energy_val_fn(theta_pred_batch, data_batch):
    """Evaluate E(θ_pred) for a batch of training points."""
    errors = []
    n_eval = min(4, theta_pred_batch.shape[0])
    for i in range(n_eval):
        tp = theta_pred_batch[i].cpu().numpy()
        h_val = float(data_batch[i].h_value) if hasattr(data_batch, '__getitem__') else float(dataset[i].h_value)
        e_ex = float(data_batch[i].e_exact) if hasattr(data_batch, '__getitem__') else float(dataset[i].e_exact)
        lat_h = make_lattice('chain_1d', N, J=J, h=h_val)
        H = builder.build(lat_h)
        bound = qc.assign_parameters(tp)
        e_pred = float(estimator.run([(bound, H)]).result()[0].data.evs)
        errors.append(abs(e_pred - e_ex))
    return errors

# Train MPNN
model = MPNNPredictor(node_features=2, hidden_dim=64, n_layers=3, output_dim=2*p_layers)
result = train_mpnn(
    model, dataset,
    n_epochs=4000, lr=1e-3, patience=150,
    energy_val_interval=50,
    energy_val_fn=energy_val_fn,
)
print(f'\nTraining complete: MSE={result["final_mse"]:.2e}, stopped_early={result["stopped_early"]}')

INFO:poc.v6.mpnn_predictor:  Epoch 200: MSE=3.75e-02, ΔE_mean=5.19e-01
INFO:poc.v6.mpnn_predictor:  Epoch 400: MSE=7.40e-02, ΔE_mean=1.75e-01
INFO:poc.v6.mpnn_predictor:  Epoch 600: MSE=4.55e-02, ΔE_mean=1.99e-01
INFO:poc.v6.mpnn_predictor:  Epoch 800: MSE=5.50e-02, ΔE_mean=5.26e-01
INFO:poc.v6.mpnn_predictor:  Epoch 1000: MSE=1.94e-02, ΔE_mean=8.21e-02
INFO:poc.v6.mpnn_predictor:  Epoch 1200: MSE=4.32e-02, ΔE_mean=1.23e-01
INFO:poc.v6.mpnn_predictor:  Epoch 1400: MSE=4.18e-02, ΔE_mean=9.41e-02
INFO:poc.v6.mpnn_predictor:  Epoch 1600: MSE=2.76e-02, ΔE_mean=5.06e-02
INFO:poc.v6.mpnn_predictor:  Epoch 1800: MSE=3.15e-02, ΔE_mean=4.38e-02
INFO:poc.v6.mpnn_predictor:  Epoch 2000: MSE=2.10e-02, ΔE_mean=7.10e-02
INFO:poc.v6.mpnn_predictor:  Epoch 2200: MSE=1.30e-02, ΔE_mean=6.99e-02
INFO:poc.v6.mpnn_predictor:  Epoch 2400: MSE=2.98e-02, ΔE_mean=7.19e-02
INFO:poc.v6.mpnn_predictor:  Epoch 2600: MSE=1.86e-02, ΔE_mean=6.23e-02
INFO:poc.v6.mpnn_predictor:  Epoch 2800: MSE=1.49e-02, ΔE_mean=6.40e


Training complete: MSE=2.99e-02, stopped_early=False


## Phase 4: Dual-Route Deployment

In [5]:
# Test point: h=1.25 (unseen, near critical region)
h_test = 1.25
solver = ClassicalSolver()
lat_test = make_lattice('chain_1d', N, J=J, h=h_test)
H_test = builder.build(lat_test)
exact_test = solver.solve(H_test, lat_test)

# MPNN prediction
model.eval()
from torch_geometric.data import Data
edge_idx, coord = builder.build_graph_data(base_lattice)
x_test = torch.tensor(
    np.stack([np.full(N, h_test), coord.astype(float)], axis=1),
    dtype=torch.float32
)
test_data = Data(x=x_test, edge_index=torch.tensor(edge_idx, dtype=torch.long))
with torch.no_grad():
    theta_pred = model(test_data).numpy().flatten()
print(f'MPNN θ_pred for h={h_test}: {theta_pred}')

MPNN θ_pred for h=1.25: [-2.9357395 -2.461541  -2.797845  -2.8145318]


In [6]:
# Observable locality check
ops_x, ops_zz = builder.build_local_observables(base_lattice)
assert_observable_locality(ops_x + ops_zz, base_lattice.edges)
print('Observable locality: PASSED')

# Route 1: Adapt-VQE
deployer = HardwareDeployer()
adapt_result = deployer.deploy_adapt_vqe(qc, H_test, theta_pred, lat_test, exact_test)

print(f'\n=== ADAPT-VQE RESULTS (h={h_test}) ===')
print(f'  ΔE/gap:     {adapt_result.delta_e_over_gap*100:.2f}%  {"✅" if adapt_result.metrics_checklist["delta_e_over_gap_lt_5pct"] else "❌"}')
print(f'  ⟨X⟩ error:  {adapt_result.mag_x_error:.2e}  {"✅" if adapt_result.metrics_checklist["mag_x_error_lt_1e-2"] else "❌"}')
print(f'  ⟨ZZ⟩ error: {adapt_result.corr_zz_error:.2e}  {"✅" if adapt_result.metrics_checklist["corr_zz_error_lt_1e-2"] else "❌"}')
print(f'  ΔE:         {adapt_result.delta_e:.2e}  {"✅" if adapt_result.metrics_checklist["delta_e_lt_1e-2"] else "❌"}')
print(f'  Fidelity:   {adapt_result.fidelity:.4f}  {"✅" if adapt_result.metrics_checklist["fidelity_gte_995"] else "❌"}')
print(f'  ADAPT iters: {adapt_result.adapt_iterations}  {"✅" if adapt_result.metrics_checklist["adapt_iterations_lte_2"] else "❌"}')
print(f'  Phase:      {adapt_result.phase_label}')
n_pass = sum(adapt_result.metrics_checklist.values())
print(f'  Checklist:  {n_pass}/6')

INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:--- Iteration #1 ---
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:Found maximum gradient 0.20045654809518204 at index 4
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:Adding new operator to the ansatz: SparsePauliOp(['ZZIIII'],
              coeffs=[1.+0.j])
INFO:qiskit_algorithms.minimum_eigensolvers.vqe:Optimization complete in 0.01305389404296875 seconds.
Found optimal point [0.02265931]
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:Current eigenvalue: -8.480391202294575
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:--- Iteration #2 ---
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:Found maximum gradient 0.20045654809518135 at index 0
INFO:qiskit_algorithms.minimum_eigensolvers.adapt_vqe:Adding new operator to the ansatz: SparsePauliOp(['IIIIZZ'],
              coeffs=[1.+0.j])
INFO:qiskit_algorithms.minimum_eigensolvers.vqe:Optimization complete in 0.017698049545288086 seconds.
Found optim

Observable locality: PASSED

=== ADAPT-VQE RESULTS (h=1.25) ===
  ΔE/gap:     4.70%  ✅
  ⟨X⟩ error:  1.05e-02  ❌
  ⟨ZZ⟩ error: 2.42e-02  ❌
  ΔE:         4.19e-02  ❌
  Fidelity:   0.9913  ❌
  ADAPT iters: 2  ✅
  Phase:      paramagnetic
  Checklist:  2/6


In [7]:
# Route 2: QRC Fallback
qrc = QRCPipeline(seed=42)
qrc.build_reservoir(N, p_layers, base_lattice)
qrc.train_readout(h_values, mag_x_exact, corr_zz_exact)

qrc_result = deployer.deploy_qrc(qrc, h_test, exact_test)

print(f'\n=== QRC RESULTS (h={h_test}) ===')
print(f'  ⟨X⟩ pred:   {qrc_result.mag_x_pred:.4f} (exact: {exact_test.mag_x:.4f})')
print(f'  ⟨ZZ⟩ pred:  {qrc_result.corr_zz_pred:.4f} (exact: {exact_test.corr_zz:.4f})')
print(f'  ⟨X⟩ error:  {qrc_result.mag_x_error:.2e}')
print(f'  ⟨ZZ⟩ error: {qrc_result.corr_zz_error:.2e}')
print(f'  Phase:      {qrc_result.phase_label}')

INFO:poc.v6.qrc_pipeline:QRC reservoir built: 6 qubits, p=2, 4 fixed params
INFO:poc.v6.qrc_pipeline:QRC readout trained: R²=0.9737 on 27 points



=== QRC RESULTS (h=1.25) ===
  ⟨X⟩ pred:   0.9028 (exact: 0.8570)
  ⟨ZZ⟩ pred:  0.4016 (exact: 0.4194)
  ⟨X⟩ error:  4.58e-02
  ⟨ZZ⟩ error: 1.78e-02
  Phase:      paramagnetic


In [8]:
# Data-driven critical point
h_c = HardwareDeployer.find_critical_point(h_values, mag_x_exact, corr_zz_exact)
print(f'\nData-driven critical point: h_c = {h_c:.3f}')
print(f'(Thermodynamic limit: h_c = 1.0)')


Data-driven critical point: h_c = 0.815
(Thermodynamic limit: h_c = 1.0)
